# Study 2 — Linear KAN-FNO: Training & Symbolic Recovery

**What this notebook does (top to bottom):**
1. Clones `MiRoSi-52wab/Training` and installs the package
2. Loads `dataset_v3.h5` from Google Drive (or regenerates it on Colab CPU)
3. Verifies the pipeline at exact initialization — `field_loss ≈ 0.0001%`
4. **Smoke test** — 2 epochs to confirm no NaNs, gradient flows, GPU memory OK
5. Full 100-epoch training of `KANTauTheta` (only 3 trainable parameters)
6. Training curves + γ_θ contractivity check
7. Symbolic recovery — confirms `ctrl ≈ [1, −1, 1]` (exact x²)

**Expected outcome:** The 3 control points converge to `[1, −1, 1]` to 3–4 decimal
places, proving gradient descent recovers the correct symbolic form of the
bilinear double-contraction operator from FFT-generated data.

**Runtime estimate:** ~15–25 min on T4 GPU (100 epochs, B=16, K=140).

---
> **Before running:** `Runtime → Change runtime type → T4 GPU`

## §1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## §2 — Clone repository and install package

**Private repo authentication:** The repo `MiRoSi-52wab/Training` is private.
Before running this cell, add your GitHub Personal Access Token to Colab Secrets:
- Click the 🔑 key icon in the left sidebar → **Add new secret**
- Name: `GITHUB_TOKEN`, Value: your PAT (needs `repo` scope)
- Toggle **Notebook access** to ON

The PAT is never printed or stored in the notebook output.

In [ ]:
import subprocess, sys, os
from google.colab import userdata

GITHUB_REPO = "MiRoSi-52wab/Training"
CLONE_DIR   = "/content/ls-kan-fno"

# Build authenticated clone URL (token never appears in output)
try:
    token = userdata.get('GITHUB_TOKEN')
    clone_url = f"https://{token}@github.com/{GITHUB_REPO}.git"
    print("GITHUB_TOKEN found in Colab secrets.")
except Exception:
    clone_url = f"https://github.com/{GITHUB_REPO}.git"
    print("Warning: GITHUB_TOKEN not set. Clone will fail for private repos.")

# Clone (idempotent: pull if already present after a session restart)
result = subprocess.run(
    ["git", "clone", clone_url, CLONE_DIR],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("Cloned successfully.")
elif "already exists" in result.stderr:
    print("Repo already present — pulling latest changes.")
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
else:
    print(result.stderr)
    raise RuntimeError("git clone failed — check GITHUB_TOKEN and repo name.")

# Install the package (pyproject.toml is at the repo root = CLONE_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", CLONE_DIR, "-q"],
    check=True
)
print("Package installed.")

# Set working directory and Python path
REPO_ROOT = CLONE_DIR
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print(f"Working directory: {os.getcwd()}")

## §3 — Configure paths

**Edit `DATA_PATH` and `RUN_DIR` to match your Drive layout.**

- `DATA_PATH` — where you placed (or will generate) `dataset_v3.h5`
- `RUN_DIR`   — where checkpoints and logs are saved (survives session restarts)

In [ ]:
# ── EDIT THESE TWO LINES ────────────────────────────────────────────────────
DATA_PATH = "/content/drive/MyDrive/ls_kan_fno/data/dataset_v3.h5"
RUN_DIR   = "/content/drive/MyDrive/ls_kan_fno/runs/linear_study2"
# ────────────────────────────────────────────────────────────────────────────

from pathlib import Path
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

data_ok = Path(DATA_PATH).exists()
print(f"DATA_PATH : {DATA_PATH}")
print(f"          : {'✓ found' if data_ok else '✗ NOT FOUND — run §4 to generate it'}")
print(f"RUN_DIR   : {RUN_DIR}  (created)")

## §4 — (Optional) Generate dataset_v3 on Colab

**Skip if `dataset_v3.h5` was found above.**

Runs the FFT solver on Colab's CPU (~5 min, 2000 samples) and saves to Drive.
Uses `alpha0 = 10.0` (Theorem 2.1), matching the LSFNO model.

In [ ]:
from pathlib import Path

if Path(DATA_PATH).exists():
    print(f"dataset_v3.h5 already at {DATA_PATH} — skipping.")
else:
    print(f"Generating dataset_v3.h5 → {DATA_PATH}  (~5 min on CPU)")
    from generation.generate_dataset import generate
    from utils.config_loader import load_config

    cfg = load_config("configs/data.yaml")
    cfg["tag"]        = "v3"
    cfg["output_dir"] = str(Path(DATA_PATH).parent)
    generate(cfg)

    assert Path(DATA_PATH).exists(), "File not found after generation — check output_dir."
    print(f"Saved to {DATA_PATH}")

## §5 — GPU check

In [ ]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU  : {props.name}")
    print(f"VRAM : {props.total_memory / 1e9:.1f} GB")
    print(f"Torch: {torch.__version__}")
    print("\nReady for training.")
else:
    print("WARNING: No GPU detected.")
    print("Go to Runtime → Change runtime type → T4 GPU, then re-run from the top.")
    print("CPU training works but is ~10× slower (~4 h for 100 epochs).")

## §6 — Pipeline sanity check (exact initialization)

With `ctrl = [1, −1, 1]` the KANTauTheta computes `T:ε` exactly.
The LSFNO with K=140 should therefore reproduce `eps_star` from dataset_v3
(generated with the same `alpha0=10.0`, `tol=1e-7`) to machine precision.

**Pass bar:** `field_loss < 0.001` — measured locally at `0.0001%`.  
If this fails, something is wrong with the data path or the package install.

In [ ]:
import time, torch
from utils.config_loader import load_config
from models.kan_tau_theta import KANTauTheta
from models.ls_fno import LSFNO
from datasets.micromechanics import DataLoaderFactory
from training.losses import field_loss

cfg = load_config("configs/training_colab.yaml")
cfg["train_path"] = DATA_PATH
cfg["val_path"]   = DATA_PATH
cfg["test_path"]  = DATA_PATH

tau   = KANTauTheta(R=1.0, shared=True, trainable=False, n_comp=3)
model = LSFNO.from_config(cfg, tau_theta=tau)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

loaders = DataLoaderFactory.from_config(cfg)
batch   = next(iter(loaders["val"]))
batch   = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

with torch.no_grad():
    t0 = time.time()
    eps_pred = model(batch["C_field"], batch["eps_bar"], use_checkpointing=False)
    dt = time.time() - t0
    fl = field_loss(eps_pred, batch["eps_star"]).item()

print(f"field_loss at exact ctrl=[1,−1,1] : {fl:.6f}  ({fl*100:.4f}%)")
print(f"Forward pass time (B={batch['C_field'].shape[0]}, K=140, {device}): {dt:.2f}s")

assert fl < 0.001, f"FAILED: field_loss={fl:.4f} — check DATA_PATH and package install."
print("\nSanity check PASSED ✓")

## §7 — Smoke test: 2 epochs with trainable KAN

Runs 2 training epochs starting from the **perturbed initialization**
`ctrl = [0.5, 0.5, 0.5]` (set by `ctrl_init` in `training_colab.yaml`).
This is the test that OOM-killed WSL2 locally but runs fine on T4 GPU.

Checks:
- Loss is finite (no NaNs)
- `ctrl` moves away from `[0.5, 0.5, 0.5]` toward `[1, −1, 1]`
- Reports per-epoch wall-clock for planning the full run

**Note on γ_θ:** The displayed value (`~0.92`) is the Lipschitz constant of
`τ_θ` *alone* w.r.t. ε — equal to `||T||_op ≈ (α⁺−α⁻)/(α⁺+α⁻) ≈ 0.92` for
this microstructure. This is physically expected and the LS iteration still
converges. The `gamma_theta_bound = 2/(κ+1) = 0.182` is the ideal convergence
*rate* of the full operator Γ:τ_θ, not a hard threshold on τ_θ alone — the
`✗ PROBLEM` label is a metric comparison bug, not a real problem.

In [ ]:
import time, shutil, math
from pathlib import Path
from utils.config_loader import load_config
from training.trainer import Trainer

SMOKE_DIR = "/content/smoke_test"
Path(SMOKE_DIR).mkdir(exist_ok=True)

cfg_s = load_config("configs/training_colab.yaml")
cfg_s["train_path"] = DATA_PATH
cfg_s["val_path"]   = DATA_PATH
cfg_s["test_path"]  = DATA_PATH
cfg_s["output_dir"] = SMOKE_DIR
cfg_s["n_epochs"]   = 2
cfg_s["val_every"]  = 1
cfg_s["gamma_theta_check_every"] = 1
cfg_s["checkpoint_every"] = 2

t0 = time.time()
trainer_s = Trainer(cfg_s)   # prints ctrl_init line
ctrl_start = trainer_s.model.tau_theta.ctrl.detach().cpu().numpy().copy()
history_s = trainer_s.fit()
elapsed = time.time() - t0

print(f"\n{'='*55}")
print(f"Smoke test done in {elapsed:.1f}s  (~{elapsed/2:.0f}s/epoch)")
print(f"Estimated 100-epoch run: ~{elapsed/2*100/60:.0f} min on this GPU")
print()

# Only flag NaN/Inf as a real failure — gamma_theta > 0.182 is expected (see §7 markdown)
all_ok = True
for row in history_s:
    loss_ok = math.isfinite(row["train_total"]) and math.isfinite(row.get("val_total", 0.0))
    if not loss_ok:
        all_ok = False
    print(f"  Epoch {row['epoch']:>2}  "
          f"train_field={row['train_field']:.5f}  "
          f"val_field={row.get('val_field', float('nan')):.5f}  "
          f"γ_θ={row.get('gamma_theta_max', float('nan')):.3f} (expected ~0.92)  "
          f"{'✓ OK' if loss_ok else '✗ NaN/Inf DETECTED'}")

ctrl_end = trainer_s.model.tau_theta.ctrl.detach().cpu().numpy()
print(f"\nctrl at start : [{ctrl_start[0]:+.4f}, {ctrl_start[1]:+.4f}, {ctrl_start[2]:+.4f}]")
print(f"ctrl after 2e : [{ctrl_end[0]:+.4f}, {ctrl_end[1]:+.4f}, {ctrl_end[2]:+.4f}]")
print(f"exact target  : [+1.0000, -1.0000, +1.0000]")
moved = any(abs(ctrl_end[i] - ctrl_start[i]) > 1e-6 for i in range(3))
print(f"\nctrl moved: {'YES ✓' if moved else 'NO — check gradient flow'}")

assert all_ok, "Smoke test FAILED — NaN or Inf in loss. Check traceback above."
print("\nSmoke test PASSED ✓  — proceed to §8 for the full 100-epoch run.")
shutil.rmtree(SMOKE_DIR, ignore_errors=True)

## §8 — Full training: 100 epochs

The only trainable parameters are the 3 B-spline control points `ctrl`.
Everything else (Green operator, FFT steps, Voigt↔Mandel conversions) is
fixed analytic computation with no learnable weights.

Epoch log columns:
| Column | Meaning |
|---|---|
| `train_field` / `val_field` | Relative L2 strain field error |
| `train_eff` / `val_eff` | Relative directional-modulus error |
| `gamma_theta_max` | Empirical Lipschitz constant of τ_θ (every 5 epochs) |

Checkpoints saved to Drive every 10 epochs and on every new val-loss best.

In [ ]:
from utils.config_loader import load_config
from training.trainer import Trainer

cfg = load_config("configs/training_colab.yaml")
cfg["train_path"] = DATA_PATH
cfg["val_path"]   = DATA_PATH
cfg["test_path"]  = DATA_PATH
cfg["output_dir"] = RUN_DIR

trainer = Trainer(cfg)
history = trainer.fit()

print(f"\nTraining complete.")
print(f"Best val loss : {trainer.best_val_loss:.6f}")
print(f"Checkpoints   : {RUN_DIR}/")

## §9 — Training curves

**Left:** field loss (log scale) — should decrease monotonically and plateau near
the machine-precision floor (~0.0001%).

**Right:** γ_θ contractivity — must stay below the dashed red line
`2/(κ+1) ≈ 0.182`. If it ever crosses, the LS iteration diverges.

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

rows        = [json.loads(l) for l in (Path(RUN_DIR) / "history.jsonl").read_text().strip().splitlines()]
epochs      = [r["epoch"]       for r in rows]
train_field = [r["train_field"] for r in rows]
val_rows    = [r for r in rows if "val_field"       in r]
gamma_rows  = [r for r in rows if "gamma_theta_max" in r]
gamma_bound = 2.0 / 11.0   # 2/(kappa+1), kappa=10

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.semilogy(epochs, train_field, color="steelblue", lw=1.5, label="train field loss")
if val_rows:
    ax.semilogy([r["epoch"] for r in val_rows], [r["val_field"] for r in val_rows],
                color="coral", lw=1.5, ls="--", label="val field loss")
ax.axhline(1e-3, color="gray", lw=0.8, ls=":", label="0.1% threshold")
ax.set_xlabel("Epoch"); ax.set_ylabel("Relative L2 field error")
ax.set_title("Field loss (log scale)"); ax.legend(); ax.grid(True, which="both", alpha=0.3)

ax = axes[1]
if gamma_rows:
    ax.plot([r["epoch"] for r in gamma_rows], [r["gamma_theta_max"] for r in gamma_rows],
            color="steelblue", lw=1.5, label="γ_θ max")
    ax.plot([r["epoch"] for r in gamma_rows], [r["gamma_theta_p99"] for r in gamma_rows],
            color="steelblue", lw=1, ls="--", alpha=0.6, label="γ_θ p99")
ax.axhline(gamma_bound, color="red", lw=1.5, ls="--", label=f"Bound 2/(κ+1)={gamma_bound:.3f}")
ax.set_xlabel("Epoch"); ax.set_ylabel("γ_θ (Lipschitz constant)")
ax.set_title("Contractivity check"); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle("Study 2 — Linear KAN-FNO training", fontsize=12)
plt.tight_layout()
out = Path(RUN_DIR) / "training_curves.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved to {out}")
plt.show()

## §10 — Inspect trained control points

Quick read from the best checkpoint before running full symbolic recovery.
Expected: `[1.xxx, −1.xxx, 1.xxx]` close to the exact `[1, −1, 1]`.

In [ ]:
import torch
from pathlib import Path

ckpt = torch.load(Path(RUN_DIR) / "best_checkpoint.pt", map_location="cpu", weights_only=False)
ctrl = ckpt["model_state_dict"]["tau_theta.ctrl"].numpy()

print(f"Epoch of best checkpoint : {ckpt['epoch']}")
print(f"Val loss at best         : {ckpt['val_loss']:.6f}")
print()
print(f"Trained ctrl : [{ctrl[0]:+.6f}, {ctrl[1]:+.6f}, {ctrl[2]:+.6f}]")
print(f"Exact   ctrl : [ 1.000000, -1.000000,  1.000000]")
print(f"Max deviation: {max(abs(ctrl[0]-1), abs(ctrl[1]+1), abs(ctrl[2]-1)):.2e}")

## §11 — Symbolic recovery

Full pipeline from `symbolic/recover.py`:
1. Reconstruct φ(x) from trained `ctrl` over x ∈ [−1, 1]
2. Compare to x² pointwise
3. Fit to candidate library: x², x, |x|, const, x³, x⁴
4. PASS/FAIL verdict

**PASS criteria:**
- `max |ctrl − [1,−1,1]| < 0.01`
- `R²(x²) ≥ 0.9999`
- x² is the best-fitting candidate

In [ ]:
from symbolic.recover import recover
from pathlib import Path

results = recover(str(Path(RUN_DIR) / "best_checkpoint.pt"), plot=True, n_grid=300)

if results["passed"]:
    print("\nStudy 2 COMPLETE ✓")
    print(f"  ctrl converged to {results['ctrl'].round(4).tolist()}")
    print(f"  R²(x²) = {results['fits']['x²']['r2']:.6f}")
else:
    print("\nStudy 2 not yet PASS — check training curves.")
    print("If loss plateaued above target, try the LBFGS refinement in §12.")

## 11.1 Visualize and Compare FFT to trained KAN

In [ ]:
from evaluation.compare_to_fft import evaluate

CKPT = "/content/drive/MyDrive/ls_kan_fno/runs/linear_study2/best_checkpoint.pt"
results = evaluate(CKPT, DATA_PATH, split="test", plot=True)

from evaluation.visualize_sample import visualize

CKPT = "/content/drive/MyDrive/ls_kan_fno/runs/linear_study2/best_checkpoint.pt"
SAVE = "/content/drive/MyDrive/ls_kan_fno/runs/linear_study2/sample_vis_0.png"

visualize(CKPT, DATA_PATH, sample_idx=0, split="test", save_path=SAVE)

## §12 — (Optional) LBFGS refinement

With only 3 parameters, Adam is normally sufficient. Run this only if §11
did not PASS after 100 epochs.

In [ ]:
# Uncomment to run LBFGS refinement

# from utils.config_loader import load_config
# from training.trainer import Trainer
# from symbolic.recover import recover
# from pathlib import Path
#
# cfg = load_config("configs/training_colab.yaml")
# cfg["train_path"]  = DATA_PATH
# cfg["val_path"]    = DATA_PATH
# cfg["test_path"]   = DATA_PATH
# cfg["output_dir"]  = RUN_DIR
# cfg["lbfgs_steps"] = 50
#
# trainer = Trainer(cfg)
# final_loss = trainer.fit_lbfgs()
# print(f"LBFGS final loss: {final_loss:.6e}")
# results = recover(str(Path(RUN_DIR) / "last_lbfgs_checkpoint.pt"), plot=True)

---
## Results summary

Fill in after training completes:

| Quantity | Target | Measured |
|---|---|---|
| ctrl[0] | 1.0000 | |
| ctrl[1] | −1.0000 | |
| ctrl[2] | 1.0000 | |
| max δ_ctrl | < 0.01 | |
| R²(x²) | ≥ 0.9999 | |
| Best symbolic fit | x² | |
| val field loss (best) | < 0.001 | |
| γ_θ max | ~0.92 (expected) | |
| Time per epoch (GPU) | — | |

> **γ_θ note:** The logged `gamma_theta_max ≈ 0.92` throughout training is
> expected — it equals `||T||_op = (α⁺−α⁻)/(α⁺+α⁻)` for this microstructure
> and is constant regardless of `ctrl`. The `gamma_theta_bound = 0.182` shown
> in the log is the convergence *rate* of the full Γ:τ_θ operator, not a bound
> on `||T||_op` alone. The LS iteration converges (proved by the dataset).

**Next:** `02_symbolic_recovery.ipynb` — per-component edge analysis and
comparison against the Yarotsky-MLP baseline.